# Étape 1 : Collecte des données
## Projet : Analyser les transactions pour détecter la fraude bancaire

**Objectif de cette étape :** récupérer le jeu de données, le stocker proprement, vérifier qu'il est complet et conforme, puis documenter sa provenance.

**Règle d'or :** les données brutes (`data/raw/`) ne se modifient **jamais**. Tous les traitements se feront sur des copies.

## 1. Présentation de la source

| Élément | Détail |
|---|---|
| Nom | Credit Card Fraud Detection |
| Fournisseur | Worldline et le Machine Learning Group de l'ULB (Université Libre de Bruxelles) |
| Lien | https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud |
| Contenu | Transactions par carte bancaire de porteurs européens |
| Période | 2 jours, septembre 2013 |
| Taille | 284 807 transactions dont 492 fraudes (≈ 0,17 %) |
| Fichier | `creditcard.csv` (environ 150 Mo) |

## 2. Préparation de l'environnement

### 2.1 Bibliothèques nécessaires



In [1]:
import hashlib
import json
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.5.3


### 2.2 Dossiers du projet

Le notebook détecte automatiquement s'il se trouve dans un dossier `notebooks/` ou à la racine du projet, et crée `data/raw/` si besoin.

In [2]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

CSV_NAME = "creditcard.csv"
CSV_PATH = RAW_DIR / CSV_NAME

print("Racine du projet :", PROJECT_ROOT)
print("Dossier des données brutes :", RAW_DIR)

Racine du projet : /home/virlence-manga-bell/Documents/DataScience/fraude-bancaire-detection
Dossier des données brutes : /home/virlence-manga-bell/Documents/DataScience/fraude-bancaire-detection/data/raw


## 3. Récupération des données

### Méthode : téléchargement manuel 

1. Créer un compte Kaggle gratuit sur https://www.kaggle.com
2. Ouvrir https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
3. Cliquer sur **Download** : tu obtiens un fichier `archive.zip`
4. Le laisser dans ton dossier *Téléchargements* ou place-le dans `data/raw/`. 

### 3.1 Localiser et préparer le fichier

Cette cellule cherche `creditcard.csv` (ou une archive `.zip` qui le contient) dans `data/raw/`, dans tes dossiers *Downloads* / *Téléchargements* et dans le dossier courant, puis le place dans `data/raw/`.

In [3]:
def chercher_et_preparer_csv():
    """Retourne un message décrivant l'action réalisée, ou None si rien n'est trouvé."""
    if CSV_PATH.exists():
        return "Le fichier est déjà présent dans data/raw/"

    dossiers = [RAW_DIR, Path.home() / "Downloads", Path.home() / "Téléchargements", Path.cwd()]
    for dossier in dossiers:
        if not dossier.exists():
            continue

        direct = dossier / CSV_NAME
        if direct.exists():
            shutil.copy(direct, CSV_PATH)
            return f"CSV copié depuis {dossier}"

        for archive in dossier.glob("*.zip"):
            try:
                with zipfile.ZipFile(archive) as z:
                    if CSV_NAME in z.namelist():
                        z.extract(CSV_NAME, RAW_DIR)
                        return f"CSV extrait de l'archive {archive}"
            except zipfile.BadZipFile:
                continue
    return None


resultat = chercher_et_preparer_csv()

if resultat is None:
    raise FileNotFoundError(
        "creditcard.csv est introuvable.\n"
        "Télécharge-le sur https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud "
        "puis relance cette cellule."
    )

print("✅", resultat)
print("Taille du fichier :", round(CSV_PATH.stat().st_size / 1024 / 1024, 1), "Mo")

✅ CSV extrait de l'archive /home/virlence-manga-bell/Documents/DataScience/fraude-bancaire-detection/data/raw/creditcard.csv.zip
Taille du fichier : 143.8 Mo


## 4. Chargement des données

In [4]:
df = pd.read_csv(CSV_PATH)

print(f"Données chargées : {df.shape[0]:,} lignes x {df.shape[1]} colonnes".replace(",", " "))
display(df.head())

Données chargées : 284 807 lignes x 31 colonnes


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 5. Contrôles de qualité de la collecte

Avant d'aller plus loin, on vérifie que ce qu'on a récupéré correspond bien à ce qui est annoncé. Un fichier tronqué, corrompu ou d'une autre version fausserait toute l'analyse sans qu'on s'en rende compte.

> Ces contrôles vérifient que la **collecte** s'est bien passée.

In [5]:
COLONNES_ATTENDUES = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]

controles = {
    "284 807 lignes": len(df) == 284_807,
    "31 colonnes": df.shape[1] == 31,
    "Noms de colonnes conformes": list(df.columns) == COLONNES_ATTENDUES,
    "Toutes les colonnes sont numériques": all(pd.api.types.is_numeric_dtype(t) for t in df.dtypes),
    "Aucune valeur manquante": int(df.isnull().sum().sum()) == 0,
    "Class ne contient que 0 et 1": set(df["Class"].unique()) == {0, 1},
    "492 fraudes": int(df["Class"].sum()) == 492,
    "Amount jamais négatif": bool((df["Amount"] >= 0).all()),
    "Time compris entre 0 et 172 792 s (48 h)": bool(df["Time"].between(0, 172_792).all()),
}

rapport = pd.DataFrame({
    "Contrôle": list(controles.keys()),
    "Résultat": ["✅ OK" if ok else "❌ ÉCHEC" for ok in controles.values()],
})
display(rapport)

if all(controles.values()):
    print("Tous les contrôles sont validés : la collecte est conforme.")
else:
    print("Au moins un contrôle a échoué : retélécharge le fichier avant de continuer.")

,Contrôle,Résultat
0,284 807 lignes,✅ OK
1,31 colonnes,✅ OK
2,Noms de colonnes conformes,✅ OK
3,Toutes les colonnes sont numériques,✅ OK
4,Aucune valeur manquante,✅ OK
5,Class ne contient que 0 et 1,✅ OK
6,492 fraudes,✅ OK
7,Amount jamais négatif,✅ OK
8,Time compris entre 0 et 172 792 s (48 h),✅ OK


Tous les contrôles sont validés : la collecte est conforme.


### Un premier constat à garder en tête

In [7]:
nb_normales = int((df["Class"] == 0).sum())
nb_fraudes = int((df["Class"] == 1).sum())

print(f"Transactions normales    : {nb_normales:,}".replace(",", " "))
print(f"Transactions frauduleuses: {nb_fraudes:,}".replace(",", " "))
print(f"Taux de fraude           : {nb_fraudes / len(df):.3%}")
print(f"Lignes en double         : {int(df.duplicated().sum())}")

Transactions normales    : 284 315
Transactions frauduleuses: 492
Taux de fraude           : 0.173%
Lignes en double         : 1081


## 6. Dictionnaire des données

| Colonne | Type | Description |
|---|---|---|
| `Time` | numérique | Secondes écoulées entre cette transaction et la première du jeu de données |
| `V1` à `V28` | numérique | Variables anonymisées issues d'une PCA. Leur signification métier n'est pas connue |
| `Amount` | numérique | Montant de la transaction (en euros) |
| `Class` | 0 ou 1 | **Variable cible** : 0 = transaction normale, 1 = fraude |

**Ce qu'il faut retenir :**
- `Class` est ce qu'on cherche à **prédire**.
- Toutes les autres colonnes sont des **variables explicatives** (features).
- Seules `Time` et `Amount` sont interprétables. 

## 7. Fiche d'identité du jeu de données (traçabilité)

Gardons une trace de **d'où viennent les données**, **quand** elles ont été collectées et **dans quel état**. L'empreinte **SHA-256** permet de vérifier plus tard que le fichier n'a pas été modifié ou corrompu.

In [8]:
def calculer_sha256(chemin, taille_bloc=1024 * 1024):
    h = hashlib.sha256()
    with open(chemin, "rb") as f:
        for bloc in iter(lambda: f.read(taille_bloc), b""):
            h.update(bloc)
    return h.hexdigest()


metadonnees = {
    "nom_dataset": "Credit Card Fraud Detection",
    "source": "https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud",
    "fournisseur": "Worldline et ULB Machine Learning Group",
    "date_collecte": datetime.now().isoformat(timespec="seconds"),
    "fichier": CSV_NAME,
    "taille_octets": int(CSV_PATH.stat().st_size),
    "sha256": calculer_sha256(CSV_PATH),
    "nb_lignes": int(len(df)),
    "nb_colonnes": int(df.shape[1]),
    "nb_fraudes": nb_fraudes,
    "taux_fraude_pct": round(nb_fraudes / len(df) * 100, 4),
    "controles_qualite": {nom: bool(ok) for nom, ok in controles.items()},
}

chemin_fiche = RAW_DIR / "creditcard_fiche_identite.json"
with open(chemin_fiche, "w", encoding="utf-8") as f:
    json.dump(metadonnees, f, ensure_ascii=False, indent=2)

print("Fiche enregistrée :", chemin_fiche)
print("SHA-256 :", metadonnees["sha256"])

Fiche enregistrée : /home/virlence-manga-bell/Documents/DataScience/fraude-bancaire-detection/data/raw/creditcard_fiche_identite.json
SHA-256 : 76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d8ee38f4b4b98551a89


## 8. Limites de ces données

- Seulement 2 jours : pas de saisonnalité (fêtes, fin de mois...).
- Cartes européennes de 2013 : les habitudes de paiement et les méthodes de fraude ont évolué depuis.
- Variables `V1`-`V28` non interprétables : difficile d'expliquer une décision du modèle en termes métier.
- Peu de fraudes (492) : peu d'exemples pour apprendre.